# Text Extraction
 Extracting text from PDF using Granite Doclings.

Input: PDF | 
Output: Markdown in txt format (granite_docling_output.txt)

Alternatives:
Proprietary - Gemini | 
Open Source - Gemma3, NanonetsOCR2, LightsOnOCR


In [1]:
import torch
from transformers import AutoProcessor,AutoModelForImageTextToText

model_name="ibm-granite/granite-docling-258M"
device="cuda" if torch.cuda.is_available() else "cpu"

#Load Preprocessor and Model
granite_doc_processor=AutoProcessor.from_pretrained(model_name)
granite_doc_model=AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=model_name,
    dtype=torch.bfloat16,
).to(device)

In [2]:
#Convert pdf files to Images using PyMuPDF

import os
import pymupdf
from PIL import Image
import io

def pdf_to_images(folder_path, dpi=300, output_folder="../data/page_images"):
    image_paths=[]
    metadata=[]
    
    os.makedirs(output_folder, exist_ok=True)
    
    for file in os.listdir(folder_path):
        if file.lower().endswith(".pdf"):
            pdf_path=os.path.join(folder_path,file)
            pdf_doc=pymupdf.open(pdf_path)
            
            for page_number, page in enumerate(pdf_doc):
                pix=page.get_pixmap(dpi=dpi)
                image_bytes=pix.tobytes("png")
                
                image=Image.open(io.BytesIO(image_bytes))
                
                image_filename = f"{os.path.splitext(file)[0]}_page_{page_number}.png"
                image_path = os.path.join(output_folder, image_filename)

                # NEW: unique filename for each page
                image_filename = f"{os.path.splitext(file)[0]}_page_{page_number}.png"
                image_path = os.path.join(output_folder, image_filename)

                # SAVE IMAGE TO DISK
                image.save(image_path)

                # STORE PATH INSTEAD OF IMAGE
                image_paths.append(image_path)
                metadata.append((file,page_number))
                
                del image
                
    return image_paths,metadata

In [3]:
#Generating images:
pdf_folder="../data/documents"
image_paths, metadata = pdf_to_images(pdf_folder)
print(len(image_paths), "pages loaded")


654 pages loaded


In [4]:
#Prompt
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."},
        ],
    },
]

prompt=granite_doc_processor.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True
)

In [28]:
# Takes around 60 seconds per batch

batch_size=5

import os
import time

progress_file = "../data/extracted_text/doctags_progress.txt"

all_doc_tags = []

# Load existing progress
if os.path.exists(progress_file):

    with open(progress_file, "r", encoding="utf-8") as f:
        content = f.read()

    all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]

    print(f"Resumed from {len(all_doc_tags)} processed pages")

start_page = len(all_doc_tags)
start_total = time.time()

# Track pages added since last save
pages_since_last_save = 0

for i in range(start_page, len(image_paths),batch_size):

    batch_start = time.time()

    batch_paths = image_paths[i:i+batch_size]
    batch_images = [Image.open(p) for p in batch_paths]

    inputs=granite_doc_processor(
        text=[prompt]*len(batch_images),
        images=batch_images,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.inference_mode():

        generated_ids = granite_doc_model.generate(
            **inputs,
            max_new_tokens=4096,
            use_cache=True
        )

    prompt_length = inputs.input_ids.shape[1]
    trimmed_generated_ids = generated_ids[:, prompt_length:]

    doctags = granite_doc_processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=False,
    )

    outputs = [o.lstrip() for o in doctags]

    all_doc_tags.extend(outputs)
    pages_since_last_save += len(outputs)

    batch_time = time.time() - batch_start
    total_time = time.time() - start_total

    print(
        f"Processed pages {i+1} → {i+len(batch_images)} | "
        f"Batch time: {batch_time:.2f}s | "
        f"Total time: {total_time/60:.2f} min"
    )

    # periodically clear GPU cache
    if (i // batch_size) % 10 == 0:
        torch.cuda.empty_cache()

    # save every 50 NEW pages
    if pages_since_last_save >= 50:

        with open(progress_file, "w", encoding="utf-8") as f:
            for tag in all_doc_tags:
                f.write(tag+"\n<<PAGE_BREAK>>\n")

        print(f"Saved progress at {len(all_doc_tags)} pages")

        pages_since_last_save = 0

    # free memory
    del batch_images
    del inputs
    del generated_ids


# FINAL SAVE (for remaining pages <50)
with open(progress_file, "w", encoding="utf-8") as f:
    for tag in all_doc_tags:
        f.write(tag+"\n<<PAGE_BREAK>>\n")

print(f"Final save completed. Total pages saved: {len(all_doc_tags)}")

Resumed from 636 processed pages
Processed pages 637 → 641 | Batch time: 202.27s | Total time: 3.37 min
Processed pages 642 → 646 | Batch time: 195.94s | Total time: 6.64 min
Processed pages 647 → 651 | Batch time: 203.06s | Total time: 10.02 min
Processed pages 652 → 654 | Batch time: 150.25s | Total time: 12.53 min
Final save completed. Total pages saved: 654


In [29]:
progress_file = "../data/extracted_text/doctags_progress.txt"

with open(progress_file, "r", encoding="utf-8") as f:
    content = f.read()

pages = [p for p in content.split("<<PAGE_BREAK>>") if p.strip()]

print("Total pages saved in doctags_progress.txt:", len(pages))

Total pages saved in doctags_progress.txt: 654


In [30]:
# from IPython.display import Markdown, display
# from docling_core.types.doc.document import DoclingDocument, DocTagsDocument
# from PIL import Image

# reconstruction_images = [Image.open(p) for p in image_paths[:len(all_doc_tags)]]

# doctag_document = DocTagsDocument.from_doctags_and_image_pairs(
#     doctags=all_doc_tags, images=reconstruction_images[:len(all_doc_tags)]
# )
# docling_document = DoclingDocument.load_from_doctags(
#     doctag_document=doctag_document, document_name="Document"
# )
# extracted_text_markdown = docling_document.export_to_markdown()
# display(Markdown(extracted_text_markdown))
from IPython.display import Markdown, display
from docling_core.types.doc.document import DoclingDocument, DocTagsDocument
from PIL import Image

progress_file = "../data/extracted_text/doctags_progress.txt"

#Read doctags file
with open(progress_file, "r", encoding="utf-8") as f:
    content = f.read()

all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]

chunk_size = 50
all_markdown = []

for i in range(0, len(all_doc_tags), chunk_size):

    chunk_tags = all_doc_tags[i:i+chunk_size]

    # load exactly same number of images as doctags
    chunk_images = [Image.open(p) for p in image_paths[i:i+len(chunk_tags)]]

    doctag_document = DocTagsDocument.from_doctags_and_image_pairs(
        doctags=chunk_tags,
        images=chunk_images
    )

    docling_document = DoclingDocument.load_from_doctags(
        doctag_document=doctag_document,
        document_name=f"Document_part_{i}"
    )

    markdown_part = docling_document.export_to_markdown()
    all_markdown.append(markdown_part)

    del chunk_images
    del doctag_document
    del docling_document

#Combine all markdown parts
extracted_text_markdown = "\n\n".join(all_markdown)

# display(Markdown(extracted_text_markdown))

In [32]:
progress_file = "../data/extracted_text/doctags_progress.txt"

with open(progress_file, "r", encoding="utf-8") as f:
    content = f.read()

pages_processed = content.count("<<PAGE_BREAK>>")

print("Pages extracted:", pages_processed)

Pages extracted: 654


In [33]:
print("Pages currently in RAM:", len(all_doc_tags))

Pages currently in RAM: 654


In [34]:
import os

output_path = "../data/extracted_text/granite_docling_output.txt"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(extracted_text_markdown)

print(f"Saved extracted text to {output_path}")

Saved extracted text to ../data/extracted_text/granite_docling_output.txt
